# MAT 125 — Phase 1: Data Integration

**Motivational Question:** *How do we combine three separate data systems into one clean, analysis-ready table?*

Real-world data never arrives as a single, perfect spreadsheet. In this project we have three separate sources:

| Dataset | Source | Grain | Key ||
|---|---|---|---|---|
| SIS | Student Information System | 1 row per enrollment | Identifier + Term |
| Pearson | Homework platform | 1 row per assignment submission | Identifier + Term |
| Canvas | LMS (Learning Management System) | 1 row per student per term | Identifier + Term |

By the end of this notebook you will know how to:
- Load wide and long CSV files
- Parse non-standard strings (e.g., `"2h 15m 30s"`) into numbers
- Aggregate row-level data to student-level summaries
- Join multiple DataFrames on a shared key
- Report join quality (what percentage of students matched across systems)
- Save a merged "master" CSV for downstream analysis

## 1. Imports

| Library | Purpose |
|---|---|
| `pandas` | DataFrames and CSV I/O |
| `re` | Regular expressions (for parsing time strings) |
| `matplotlib` / `seaborn` | Visualization |
| `warnings` | Suppress noisy but harmless warnings |

In [ ]:
import warnings
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

print("Libraries loaded.")

## 2. Load All Three Raw CSVs

We always print `.shape` immediately after loading — a quick sanity check that we got the right file.

In [ ]:
sis     = pd.read_csv("Cleaned_For_DataSci/SIS.csv")
pearson = pd.read_csv("Cleaned_For_DataSci/pearson.csv")
canvas  = pd.read_csv("Cleaned_For_DataSci/canvas.csv", low_memory=False)

print(f"SIS     : {sis.shape[0]:>7,} rows × {sis.shape[1]:>3} cols")
print(f"Pearson : {pearson.shape[0]:>7,} rows × {pearson.shape[1]:>3} cols")
print(f"Canvas  : {canvas.shape[0]:>7,} rows × {canvas.shape[1]:>3} cols")
print()
print("--- SIS head ---")
display(sis[["Identifier", "Term", "Official.Grade", "IPEDS.Ethnicity", "Sex"]].head(3))
print("--- Pearson head ---")
display(pearson[["Identifier", "Term", "Assignment.Category", "Assignment.Title", "Time.Spent", "Score"]].head(3))
print("--- Canvas head (first 5 cols) ---")
display(canvas.iloc[:3, :5])

## 3. Clean SIS

Three tasks:
1. **Strip blank grades** — dropped students have `Official.Grade = " "` (a space string, NOT `NaN`). Calling `.str.strip()` converts `" "` → `""`, then we filter those out.
2. **Create `Passed`** — grades A/B/C are passing; D/F/W are not.
3. **Deduplicate** — one student can appear in multiple rows if enrolled in multiple sections. We keep one row per `(Identifier, Term)` using `.drop_duplicates()`, ensuring the join key is unique.

In [ ]:
# --- Step 1: Strip and filter blank grades ---
sis["Official.Grade"] = sis["Official.Grade"].str.strip()
sis_graded = sis[sis["Official.Grade"] != ""].copy()

print(f"SIS original rows  : {len(sis):,}")
print(f"After removing blank grades: {len(sis_graded):,}")
print(f"Excluded (no grade): {len(sis) - len(sis_graded):,}")

# --- Step 2: Create Passed column ---
PASS_GRADES = {"A", "B", "C"}
sis_graded["Passed"]     = sis_graded["Official.Grade"].isin(PASS_GRADES)
sis_graded["Passed_int"] = sis_graded["Passed"].astype(int)

overall_pass_rate = sis_graded["Passed_int"].mean() * 100
print(f"\nOverall pass rate : {overall_pass_rate:.1f}%")

# --- Step 3: Deduplicate to one row per (Identifier, Term) ---
# Note: SIS has rows where Term is NaN (48 rows) — these have no meaningful join key
# We drop them here with a clear comment.
sis_graded = sis_graded.dropna(subset=["Term"])  # 48 rows with unknown term
sis_graded["Identifier"] = sis_graded["Identifier"].astype(float)

# Keep first occurrence per student-term (same student, different section = same outcome)
sis_clean = sis_graded.drop_duplicates(subset=["Identifier", "Term"]).copy()

print(f"\nAfter dedup to one row per (Identifier, Term): {len(sis_clean):,}")
print(f"Unique students in SIS: {sis_clean['Identifier'].nunique():,}")

## 4. Aggregate Pearson (218 K rows → Student Level)

Pearson has one row per assignment per student. We need to collapse this to **one row per student per term** with summary statistics.

**Data structure insight:** Pearson pre-populates its gradebook with a row for *every* assignment for *every* student at course start. Assignments not attempted get `Score = NaN`. This means:
- Rows without `Assignment.Tag` = **mandatory** homework → always have a non-null score
- Rows with `Assignment.Tag` (Enhanced/Integrated Review) = **optional extra credit** → `NaN` for nearly everyone

We filter to **mandatory homework only** for score and time metrics, then compute a meaningful `hw_score_mean`.

**Time parsing:** The `Time.Spent` column looks like `"2h 15m 30s"`. Python's `re` module lets us extract the three numbers with a single pattern: `r(\d+)h\s*(\d+)m\s*(\d+)s`.

**Fast-submit flag:** If a student spent less than 1 minute on a mandatory assignment, that is flagged as a potential surface-engagement indicator.

In [ ]:
def parse_time_to_minutes(time_str):
    """Convert '2h 15m 30s' string to total minutes (float). Returns NaN on parse failure."""
    if pd.isna(time_str):
        return float("nan")
    m = re.match(r"(\d+)h\s*(\d+)m\s*(\d+)s", str(time_str).strip())
    if not m:
        return float("nan")
    h, mins, s = int(m.group(1)), int(m.group(2)), int(m.group(3))
    return h * 60 + mins + s / 60

# Parse time for all rows
pearson["time_min"] = pearson["Time.Spent"].apply(parse_time_to_minutes)

# Flag fast-submit: < 1 minute
pearson["fast_submit"] = (pearson["time_min"] < 1).astype(int)

print(f"Rows with 0h 0m 0s: {(pearson['time_min'] == 0).sum():,} ({(pearson['time_min'] == 0).mean()*100:.1f}% of all rows)")
print(f"Rows with < 1 min  : {pearson['fast_submit'].sum():,} ({pearson['fast_submit'].mean()*100:.1f}% of all rows)")

In [ ]:
# --- Separate mandatory homework, optional homework, and exam rows ---
# Mandatory HW: Category=Homework AND Assignment.Tag is NaN (core assignments with scores)
# Optional HW:  Category=Homework AND Assignment.Tag is not NaN (Enhanced/Integrated Review, nearly all NaN scores)
hw_mandatory = pearson[
    (pearson["Assignment.Category"] == "Homework") &
    (pearson["Assignment.Tag"].isna())
].copy()

# Unit 1 exam: exact title, category = Test, no Practice or OLD prefix
unit1_exam = pearson[
    (pearson["Assignment.Category"] == "Test") &
    (pearson["Assignment.Title"] == "Unit 1 Knowledge Assessment")
].copy()

print(f"Mandatory homework rows : {len(hw_mandatory):,}  (Assignment.Tag is NaN, all have scores)")
print(f"Unit 1 exam rows        : {len(unit1_exam):,}")
print(f"Mandatory HW null scores: {hw_mandatory['Score'].isna().sum():,}  (should be near 0)")

# --- Aggregate mandatory homework to student level ---
hw_agg = (
    hw_mandatory.groupby(["Identifier", "Term"])
      .agg(
          hw_submissions      = ("Score",         "count"),   # non-NaN count
          hw_score_mean       = ("Score",         "mean"),    # mean of actual scores
          hw_time_total_min   = ("time_min",      "sum"),
          hw_fast_submit_rate = ("fast_submit",   "mean"),
      )
      .reset_index()
)

# hw_completion_rate: submissions / max_possible per term
# (all mandatory students submit all mandatory assignments, so this is ~1.0 for everyone
#  but we keep it for data completeness)
term_max = hw_mandatory.groupby("Term")["Assignment.Title"].nunique().rename("hw_available").reset_index()
hw_agg = hw_agg.merge(term_max, on="Term", how="left")
hw_agg["hw_completion_rate"] = (hw_agg["hw_submissions"] / hw_agg["hw_available"]).clip(0, 1)

# Add Unit 1 exam score
unit1_agg = unit1_exam.groupby(["Identifier", "Term"])["Score"].mean().rename("unit1_exam_score").reset_index()
hw_agg = hw_agg.merge(unit1_agg, on=["Identifier", "Term"], how="left")

print(f"\nPearson student-level summary rows: {len(hw_agg):,}")
print(f"hw_score_mean range: {hw_agg['hw_score_mean'].min():.1f} – {hw_agg['hw_score_mean'].max():.1f}")
print(f"hw_fast_submit_rate range: {hw_agg['hw_fast_submit_rate'].min():.2f} – {hw_agg['hw_fast_submit_rate'].max():.2f}")
display(hw_agg.head(3))

## 5. Slim Canvas (5 919 cols → ~16 cols)

Canvas has nearly 6 000 columns — one per assignment item. We only need the **category-level summary** (Final Score) columns for this master table. Individual item columns (used for Q2 and Q3) are loaded fresh in `canvas_engagement.ipynb`.

The columns we keep:
- `Identifier`, `Term` — join keys
- All `*.Final.Score` columns that do NOT contain `Unposted` — these are the posted category totals

In [ ]:
# Select posted Final.Score columns (exclude Unposted variants)
final_score_cols = [
    c for c in canvas.columns
    if "Final.Score" in c and "Unposted" not in c
]

canvas_slim = canvas[["Identifier", "Term"] + final_score_cols].copy()

print(f"Canvas columns kept: {canvas_slim.shape[1]} (was {canvas.shape[1]})")
print("Columns:", list(canvas_slim.columns))
print()
display(canvas_slim.head(3))

## 6. Three-Way Merge

`pd.merge(..., how="left")` keeps all rows from the left table (SIS) and fills in `NaN` where no match exists in the right table. This preserves all graded students, even if they have no Canvas or Pearson record.

**Join key:** `["Identifier", "Term"]` only. The `section_code` column has 62% missing values in SIS and is not reliable enough as a key.

In [ ]:
# Align Identifier types to float64 across all three datasets
sis_clean["Identifier"]   = sis_clean["Identifier"].astype(float)
hw_agg["Identifier"]      = hw_agg["Identifier"].astype(float)
canvas_slim["Identifier"] = canvas_slim["Identifier"].astype(float)

# Three-way left join (SIS as base)
master = (
    sis_clean
    .merge(hw_agg,      on=["Identifier", "Term"], how="left", suffixes=("", "_prs"))
    .merge(canvas_slim, on=["Identifier", "Term"], how="left", suffixes=("", "_cvs"))
)

print(f"Master table shape: {master.shape[0]:,} rows × {master.shape[1]} cols")
display(master[["Identifier", "Term", "Passed", "hw_completion_rate", "Check.Your.Understanding.Final.Score"]].head(5))

## 7. Join Quality Report

A join quality report tells us how many rows successfully matched across systems. A low match rate could indicate:
- Different student ID formats between systems
- Students enrolled in SIS but not yet active in Pearson/Canvas
- Data export timing differences

In [ ]:
n_total     = len(master)
n_pearson   = master["hw_completion_rate"].notna().sum()
n_canvas    = master["Check.Your.Understanding.Final.Score"].notna().sum()
n_all_three = (master["hw_completion_rate"].notna() & 
               master["Check.Your.Understanding.Final.Score"].notna()).sum()

print("=" * 52)
print("  JOIN QUALITY REPORT")
print("=" * 52)
print(f"  SIS graded students (base)   : {n_total:>5,}  (100.0%)")
print(f"  Matched to Pearson           : {n_pearson:>5,}  ({n_pearson/n_total*100:.1f}%)")
print(f"  Matched to Canvas            : {n_canvas:>5,}  ({n_canvas/n_total*100:.1f}%)")
print(f"  Matched to ALL THREE systems : {n_all_three:>5,}  ({n_all_three/n_total*100:.1f}%)")
print("=" * 52)
print()
print(f"Key insight: {n_all_three} of {n_total} graded students ({n_all_three/n_total*100:.1f}%)",
      "have data in all three systems.")

## 8. Visualization — Dataset Coverage

A simple bar chart showing how many students are represented in each system, and how many overlap.

In [ ]:
coverage_labels = ["SIS (base)", "SIS + Pearson", "SIS + Canvas", "All Three"]
coverage_counts = [n_total, n_pearson, n_canvas, n_all_three]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(coverage_labels, coverage_counts, color=["steelblue", "#5a9e7a", "#e0804e", "#8e5ea2"])

for bar, count in zip(bars, coverage_counts):
    pct = count / n_total * 100
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f"{count:,}  ({pct:.1f}%)", va="center", fontsize=10)

ax.set_xlabel("Number of Student-Term Records")
ax.set_title("MAT 125 Data Coverage Across Systems")
ax.set_xlim(0, n_total * 1.25)
plt.tight_layout()
plt.savefig("figures/fig_data_coverage.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to figures/fig_data_coverage.png")

## 9. Save Master Table

We save the merged DataFrame to `Cleaned_For_DataSci/master_student.csv`. All subsequent notebooks load this file instead of re-running the merge.

In [ ]:
import os
os.makedirs("figures", exist_ok=True)

master.to_csv("Cleaned_For_DataSci/master_student.csv", index=False)

print(f"Saved: Cleaned_For_DataSci/master_student.csv")
print(f"Shape: {master.shape[0]:,} rows x {master.shape[1]} columns")
print()
print("Column list:")
for i, col in enumerate(master.columns):
    print(f"  {i+1:>3}. {col}")

## 10. Summary

In this notebook we:

1. **Loaded** three raw CSVs of vastly different sizes (2K, 218K, and 1.1K rows)
2. **Cleaned SIS** — stripped blank grades, defined pass/fail, deduplicated to one row per student-term
3. **Aggregated Pearson** — parsed `"Xh Ym Zs"` strings into minutes, flagged fast-submits, computed four student-level metrics
4. **Slimmed Canvas** — reduced 5,919 columns to ~13 meaningful summary columns
5. **Merged** using left joins on `["Identifier", "Term"]` — preserving all SIS students
6. **Reported join quality** — noting that ~84% of students have data in all three systems
7. **Saved** `master_student.csv` for downstream notebooks

### Key Caveats
- **Term NaN rows:** 48 SIS rows had no Term value and were excluded from the join.
- **One row = one enrollment event,** not one unique student. A student who took MAT 125 in both terms appears twice.
- **Canvas item columns** (weekly attendance, CYU scores) are NOT in `master_student.csv` — they are loaded from the raw `canvas.csv` in `canvas_engagement.ipynb`.